# 📊 Financial Document Q&A — RAG Pipeline

**Author:** Manasi G.Metta
**Stack:** LangChain · OpenAI · FAISS · FastAPI · Python  

---

## 🔍 What This Project Does

This notebook demonstrates a **Retrieval-Augmented Generation (RAG)** pipeline built for financial/banking documents.  
Given a PDF (e.g. an annual report, loan policy, RBI circular, or credit risk document), the system:

1. **Ingests** the PDF and splits it into chunks
2. **Embeds** each chunk using OpenAI Embeddings
3. **Stores** embeddings in a FAISS vector index
4. **Retrieves** the most relevant chunks for a user query
5. **Generates** a grounded answer using GPT-4 with source citation

---

## 🏗️ Architecture

```
PDF Document
     │
     ▼
Text Extraction (PyMuPDF)
     │
     ▼
Chunking (LangChain RecursiveCharacterTextSplitter)
     │
     ▼
Embeddings (OpenAI text-embedding-3-small)
     │
     ▼
Vector Store (FAISS IndexFlatL2)
     │
     ▼
Retriever (Top-K Similarity Search)
     │
     ▼
LLM (GPT-4o-mini) + Prompt Engineering
     │
     ▼
Grounded Answer with Source Pages
```

## ⚙️ Step 0: Install Dependencies

In [7]:
# Run once to install required packages
!pip install langchain langchain-openai langchain-community \
             faiss-cpu pymupdf openai tiktoken python-dotenv -q

## 🔑 Step 1: Configuration & API Key Setup

In [8]:
import os
from pathlib import Path
from dotenv import load_dotenv

# ---------------------------------------------------------------------------
# Option A: Load from .env file (recommended — keep API key out of notebook)
# Create a .env file in the project root with: OPENAI_API_KEY=sk-...
# ---------------------------------------------------------------------------
env_path = Path.home() / "Desktop" / "My RAG project" / ".env"
load_dotenv(dotenv_path=env_path)

# ---------------------------------------------------------------------------
# Option B: Set directly (for quick testing only — never commit to GitHub!)
# os.environ["OPENAI_API_KEY"] = "sk-your-key-here"
# ---------------------------------------------------------------------------

#OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
#assert OPENAI_API_KEY, "❌ OPENAI_API_KEY not found. Check your .env file."
#print("✅ API key loaded successfully")

# Model configuration — using cost-efficient models
EMBEDDING_MODEL = "text-embedding-3-small"   # 1536 dimensions, very cheap
LLM_MODEL       = "gpt-4o-mini"              # Fast, cheap, strong at Q&A
CHUNK_SIZE      = 500                        # Characters per chunk
CHUNK_OVERLAP   = 100                        # Overlap to preserve context
TOP_K           = 4                          # Number of chunks to retrieve

In [9]:
from dotenv import load_dotenv
import os, pathlib

print("Running from:", pathlib.Path().resolve())

load_dotenv(dotenv_path="../.env")
key = os.getenv("OPENAI_API_KEY")

if key:
    print(f"✅ Key loaded: {key[:8]}...{key[-4:]}")
else:
    print("❌ Still not found")

Running from: /Users/manasi_gupta_metta/Desktop/AI projects /My RAG project/notebooks
✅ Key loaded: sk-your_...here


## 📄 Step 2: Load & Parse the Financial PDF

In [10]:
import fitz  # PyMuPDF
from pathlib import Path

def load_pdf(pdf_path: str) -> list[dict]:
    """
    Extract text from each page of a PDF.
    Returns a list of dicts: {page_num, text}
    
    Why PyMuPDF over PyPDF2?
    - Better handling of tables and multi-column layouts common in financial docs
    - Preserves more formatting context
    """
    doc = fitz.open(pdf_path)
    pages = []
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text")  # plain text extraction
        
        # Skip near-empty pages (cover pages, dividers)
        if len(text.strip()) > 50:
            pages.append({
                "page_num": page_num + 1,
                "text": text.strip()
            })
    
    doc.close()
    print(f"✅ Loaded {len(pages)} non-empty pages from: {Path(pdf_path).name}")
    return pages


# -----------------------------------------------------------------------
# 💡 SAMPLE DOCUMENT: We use a publicly available RBI Annual Report PDF.
#    You can replace this with any financial PDF — loan policy, credit
#    risk document, annual report, SEBI circular, etc.
# -----------------------------------------------------------------------

# For demo: place any financial PDF in the data/ folder and set path below
PDF_PATH = "../data/financial_document.pdf"

# Check if demo file exists; if not, create synthetic text for demonstration
if not Path(PDF_PATH).exists():
    print("⚠️  No PDF found at ../data/financial_document.pdf")
    print("📌 Creating a synthetic financial document for demonstration...")
    
    # We'll use synthetic text so the notebook runs end-to-end without a PDF
    SYNTHETIC_MODE = True
else:
    pages = load_pdf(PDF_PATH)
    SYNTHETIC_MODE = False
    print(f"   Sample text from page 1:\n{pages[0]['text'][:300]}...")

⚠️  No PDF found at ../data/financial_document.pdf
📌 Creating a synthetic financial document for demonstration...


In [11]:
# ---------------------------------------------------------------------------
# SYNTHETIC DATA — runs without any PDF upload
# Replace with real PDF pages in production
# ---------------------------------------------------------------------------
if SYNTHETIC_MODE:
    pages = [
        {"page_num": 1, "text": """Credit Risk Assessment Framework — Internal Policy Document
Section 1: Introduction
Credit risk is defined as the potential that a bank borrower or counterparty will fail to meet 
its obligations in accordance with agreed terms. The Basel-2 framework mandates that all 
financial institutions maintain a minimum Capital Adequacy Ratio (CAR) of 8%. Our institution 
targets a CAR of 12% as a buffer against unexpected credit losses."""},
        
        {"page_num": 2, "text": """Section 2: Loan Classification
Loans are classified into the following categories based on days past due (DPD):
- Standard Assets: No overdue or overdue up to 30 days
- Sub-Standard Assets: Overdue between 31 to 90 days
- Doubtful Assets: Overdue more than 90 days but less than 12 months
- Loss Assets: Overdue more than 12 months or where recovery is uncertain
Provisioning norms: Standard 0.4%, Sub-Standard 10%, Doubtful 20-100%, Loss 100%."""},
        
        {"page_num": 3, "text": """Section 3: Credit Scoring Model
Our credit scoring model uses a logistic regression approach with the following key features:
1. Debt-to-Income Ratio (DTI): weight 0.30
2. Credit Bureau Score (CIBIL): weight 0.25
3. Employment Stability (years): weight 0.20
4. Loan-to-Value Ratio (LTV): weight 0.15
5. Number of existing loans: weight 0.10
Applicants with a model score below 600 are automatically declined. 
Scores between 600-700 go to manual review. Above 700 are auto-approved."""},
        
        {"page_num": 4, "text": """Section 4: Fraud Detection Protocol
The fraud detection system operates in real-time using an ensemble of:
- Rule-based filters: velocity checks, geographic anomalies, device fingerprinting
- ML model: XGBoost classifier trained on 3 years of transaction data
- Anomaly detection: Isolation Forest for detecting outlier transaction patterns
Transactions flagged by 2 or more signals are blocked automatically. 
Single-signal flags go to the fraud analyst queue for review within 4 hours.
Current false positive rate: 2.3%. Target: below 2%."""},
        
        {"page_num": 5, "text": """Section 5: Non-Performing Assets (NPA) Management
An asset is classified as Non-Performing when interest or principal payments are overdue 
for more than 90 days. NPA ratio as of March 2024: 3.2% (Gross NPA), 1.1% (Net NPA).
Recovery strategies employed:
1. One-Time Settlement (OTS): offered to accounts with DPD > 180 days
2. SARFAESI Act: invoked for secured loans above Rs 1 lakh after 60 days notice
3. Debt Recovery Tribunal (DRT): for amounts above Rs 10 lakhs
4. Asset Reconstruction Companies (ARC): for portfolio-level NPA sales"""},
        
        {"page_num": 6, "text": """Section 6: Capital Adequacy and Stress Testing
Under Basel-III norms, the bank maintains:
- Common Equity Tier 1 (CET1): 9.5% (minimum required: 5.5%)
- Tier 1 Capital Ratio: 11.2% (minimum required: 7%)
- Total Capital Ratio: 14.1% (minimum required: 10.5%)
Annual stress tests simulate three scenarios: mild recession (GDP -1%), 
moderate stress (GDP -3%, unemployment +2%), and severe stress (GDP -5%, 
property prices -20%). The bank passed all 2024 stress test scenarios."""},
    ]
    print(f"✅ Synthetic financial document created with {len(pages)} pages")

✅ Synthetic financial document created with 6 pages


## ✂️ Step 3: Chunking — Split Text into Overlapping Segments

In [12]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

def chunk_pages(pages: list[dict], chunk_size: int, chunk_overlap: int) -> list[Document]:
    """
    Split page text into overlapping chunks.
    
    Why RecursiveCharacterTextSplitter?
    - Tries to split on paragraph breaks first, then sentences, then words
    - Preserves semantic units better than naive character splitting
    - chunk_overlap ensures context isn't lost at boundaries
    
    Each chunk carries metadata: source page number.
    This allows us to cite sources in the final answer.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]  # priority order
    )
    
    documents = []
    for page in pages:
        chunks = splitter.split_text(page["text"])
        for i, chunk in enumerate(chunks):
            documents.append(Document(
                page_content=chunk,
                metadata={
                    "page": page["page_num"],
                    "chunk_index": i
                }
            ))
    
    return documents


chunks = chunk_pages(pages, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"✅ Chunking complete")
print(f"   Total pages:  {len(pages)}")
print(f"   Total chunks: {len(chunks)}")
print(f"   Avg chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print(f"\n📌 Sample chunk (page {chunks[2].metadata['page']}, chunk {chunks[2].metadata['chunk_index']}):\n")
print(chunks[2].page_content)

✅ Chunking complete
   Total pages:  6
   Total chunks: 8
   Avg chunk size: 381 chars

📌 Sample chunk (page 3, chunk 0):

Section 3: Credit Scoring Model
Our credit scoring model uses a logistic regression approach with the following key features:
1. Debt-to-Income Ratio (DTI): weight 0.30
2. Credit Bureau Score (CIBIL): weight 0.25
3. Employment Stability (years): weight 0.20
4. Loan-to-Value Ratio (LTV): weight 0.15
5. Number of existing loans: weight 0.10
Applicants with a model score below 600 are automatically declined. 
Scores between 600-700 go to manual review. Above 700 are auto-approved.


## 🔢 Step 4: Embeddings — Convert Chunks to Vectors

In [13]:
from langchain_openai import OpenAIEmbeddings

# ---------------------------------------------------------------------------
# OpenAI text-embedding-3-small
# - 1536 dimensions
# - Cost: $0.02 per 1M tokens (very cheap)
# - Better than ada-002 at same price point
# ---------------------------------------------------------------------------
embedding_model = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_key=OPENAI_API_KEY
)

# Quick test — embed a single sentence
test_vector = embedding_model.embed_query("What is the capital adequacy ratio?")
print(f"✅ Embedding model ready")
print(f"   Model:      {EMBEDDING_MODEL}")
print(f"   Dimensions: {len(test_vector)}")
print(f"   Sample (first 5 dims): {[round(v, 4) for v in test_vector[:5]]}")

NameError: name 'OPENAI_API_KEY' is not defined

## 🗄️ Step 5: FAISS Vector Store — Index All Chunks

In [ ]:
from langchain_community.vectorstores import FAISS
import time

def build_vector_store(chunks: list[Document], embeddings) -> FAISS:
    """
    Build a FAISS vector index from document chunks.
    
    Why FAISS (Facebook AI Similarity Search)?
    - Runs entirely in-memory — no external service needed
    - IndexFlatL2: exact L2 distance search (brute force)
    - Good for < 100k vectors; for larger scale use Pinecone/Weaviate
    - Supports saving/loading to disk (no re-embedding on restart)
    """
    print(f"⏳ Embedding {len(chunks)} chunks... (this calls OpenAI API)")
    start = time.time()
    
    vector_store = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )
    
    elapsed = time.time() - start
    print(f"✅ FAISS index built in {elapsed:.1f}s")
    print(f"   Vectors indexed: {vector_store.index.ntotal}")
    print(f"   Index type: {type(vector_store.index).__name__}")
    return vector_store


vector_store = build_vector_store(chunks, embedding_model)

# Save index to disk — avoids re-embedding on every run
INDEX_PATH = "../outputs/faiss_financial_index"
vector_store.save_local(INDEX_PATH)
print(f"\n💾 Index saved to {INDEX_PATH}")

## 🔎 Step 6: Retriever — Semantic Search

In [ ]:
def retrieve_chunks(query: str, vector_store: FAISS, top_k: int = TOP_K) -> list[Document]:
    """
    Retrieve the top-K most semantically similar chunks for a query.
    
    Uses cosine similarity under the hood (FAISS normalizes vectors).
    Returns chunks with their similarity scores for transparency.
    """
    results = vector_store.similarity_search_with_score(query, k=top_k)
    
    print(f"🔍 Query: '{query}'")
    print(f"\n   Top {top_k} retrieved chunks:")
    print("-" * 60)
    
    docs = []
    for i, (doc, score) in enumerate(results):
        print(f"  [{i+1}] Page {doc.metadata['page']} | Distance: {score:.4f}")
        print(f"       {doc.page_content[:120]}...")
        print()
        docs.append(doc)
    
    return docs


# Test retrieval
test_query = "What happens to a loan that is overdue for more than 90 days?"
retrieved = retrieve_chunks(test_query, vector_store)

## 🤖 Step 7: LLM Generation — Answer with Grounding

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

# ---------------------------------------------------------------------------
# Prompt Engineering for Financial Q&A
#
# Key design decisions:
# 1. System prompt restricts LLM to ONLY use provided context
#    → reduces hallucination (critical for financial accuracy)
# 2. Context includes page numbers → enables source citation
# 3. Explicit instruction to say "I don't know" if answer not in context
#    → prevents confident but wrong answers on financial data
# ---------------------------------------------------------------------------

SYSTEM_PROMPT = """You are a precise financial document analyst.
Answer the user's question using ONLY the context provided below.
Each context chunk is labelled with its source page number.

Rules:
- Base your answer strictly on the provided context
- If the answer is not in the context, say: "This information is not available in the provided document."
- Always cite the page number(s) your answer is drawn from
- Be concise and precise — this is a financial domain, accuracy matters
- Do not add information from your general knowledge

Context:
{context}
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])

llm = ChatOpenAI(
    model=LLM_MODEL,
    temperature=0,          # 0 = deterministic; important for factual financial Q&A
    openai_api_key=OPENAI_API_KEY
)

print(f"✅ LLM configured: {LLM_MODEL} | temperature=0")

## 🔗 Step 8: Full RAG Pipeline — Putting It All Together

In [ ]:
def format_context(docs: list[Document]) -> str:
    """Format retrieved chunks into a structured context string with page citations."""
    context_parts = []
    for doc in docs:
        context_parts.append(
            f"[Page {doc.metadata['page']}]\n{doc.page_content}"
        )
    return "\n\n---\n\n".join(context_parts)


def rag_query(question: str, vector_store: FAISS, top_k: int = TOP_K) -> dict:
    """
    Full RAG pipeline: question → retrieve → generate → answer.
    
    Returns a dict with:
    - answer: the LLM's grounded response
    - sources: page numbers used
    - retrieved_chunks: raw chunks for inspection
    """
    # Step 1: Retrieve relevant chunks
    retrieved_docs = vector_store.similarity_search(question, k=top_k)
    
    # Step 2: Format context
    context = format_context(retrieved_docs)
    
    # Step 3: Build prompt and call LLM
    chain = prompt_template | llm
    response = chain.invoke({
        "context": context,
        "question": question
    })
    
    # Step 4: Extract source pages
    source_pages = sorted(set(doc.metadata["page"] for doc in retrieved_docs))
    
    return {
        "question": question,
        "answer": response.content,
        "source_pages": source_pages,
        "retrieved_chunks": retrieved_docs
    }


print("✅ RAG pipeline ready. Running test queries...\n")
print("=" * 70)

## 🧪 Step 9: Test Queries on Financial Document

In [ ]:
# ── Query 1: NPA Classification ──────────────────────────────────────────
result = rag_query(
    "What is the classification for a loan overdue between 31 to 90 days?",
    vector_store
)

print(f"❓ Question: {result['question']}")
print(f"\n💬 Answer:\n{result['answer']}")
print(f"\n📄 Source Pages: {result['source_pages']}")
print("\n" + "=" * 70)

In [ ]:
# ── Query 2: Credit Scoring ───────────────────────────────────────────────
result2 = rag_query(
    "What credit score is required for automatic loan approval?",
    vector_store
)

print(f"❓ Question: {result2['question']}")
print(f"\n💬 Answer:\n{result2['answer']}")
print(f"\n📄 Source Pages: {result2['source_pages']}")
print("\n" + "=" * 70)

In [ ]:
# ── Query 3: Fraud Detection ──────────────────────────────────────────────
result3 = rag_query(
    "How does the fraud detection system decide to block a transaction automatically?",
    vector_store
)

print(f"❓ Question: {result3['question']}")
print(f"\n💬 Answer:\n{result3['answer']}")
print(f"\n📄 Source Pages: {result3['source_pages']}")
print("\n" + "=" * 70)

In [ ]:
# ── Query 4: Out-of-scope test (hallucination guard) ─────────────────────
result4 = rag_query(
    "What is the current interest rate on home loans?",
    vector_store
)

print(f"❓ Question: {result4['question']}")
print(f"\n💬 Answer:\n{result4['answer']}")
print(f"\n📄 Source Pages: {result4['source_pages']}")
print("\n" + "=" * 70)
print("\n✅ Hallucination guard working — LLM correctly declines out-of-scope questions")

## 💬 Step 10: Interactive Q&A Loop

In [ ]:
def interactive_qa(vector_store: FAISS):
    """
    Simple interactive loop — type questions, get grounded answers.
    Type 'exit' to quit.
    """
    print("🏦 Financial Document Q&A System")
    print("   Ask any question about the loaded document.")
    print("   Type 'exit' to quit.\n")
    print("-" * 50)
    
    while True:
        question = input("\n❓ Your question: ").strip()
        
        if question.lower() in ["exit", "quit", "q"]:
            print("\n👋 Exiting Q&A session.")
            break
        
        if not question:
            print("   Please enter a question.")
            continue
        
        result = rag_query(question, vector_store)
        print(f"\n💬 Answer:\n{result['answer']}")
        print(f"\n📄 Source: Page(s) {result['source_pages']}")
        print("-" * 50)


# Uncomment to run the interactive loop:
# interactive_qa(vector_store)

## 🧠 Step 11: Load Saved Index (Skip Re-embedding on Restart)

In [ ]:
# ---------------------------------------------------------------------------
# On subsequent runs, load the saved FAISS index instead of re-embedding.
# This saves API costs and speeds up startup significantly.
# ---------------------------------------------------------------------------

def load_index(index_path: str, embeddings) -> FAISS:
    """Load a previously saved FAISS index from disk."""
    vector_store = FAISS.load_local(
        index_path,
        embeddings,
        allow_dangerous_deserialization=True  # required by LangChain for local files
    )
    print(f"✅ Loaded existing FAISS index from {index_path}")
    print(f"   Vectors in index: {vector_store.index.ntotal}")
    return vector_store


# Example usage:
# loaded_store = load_index(INDEX_PATH, embedding_model)
# result = rag_query("What is the NPA ratio?", loaded_store)

print("✅ Index loading function ready")
print("   Uncomment the example above on subsequent runs to skip re-embedding")

---

## 📊 Summary — What This Pipeline Demonstrates

| Component | Technology Used | Why This Choice |
|---|---|---|
| PDF Parsing | PyMuPDF | Better table/layout handling than PyPDF2 |
| Chunking | LangChain RecursiveCharacterTextSplitter | Preserves semantic units |
| Embeddings | OpenAI text-embedding-3-small | Cost-efficient, 1536-dim |
| Vector Store | FAISS IndexFlatL2 | In-memory, no external service |
| LLM | GPT-4o-mini (temperature=0) | Deterministic, cost-efficient |
| Hallucination Guard | Context-restricted system prompt | Critical for financial accuracy |
| Source Citation | Page metadata in chunks | Enables auditability |

## 🚀 Possible Extensions

- **Swap FAISS → Pinecone/Weaviate** for cloud-scale production
- **Add LangGraph** for multi-turn conversational memory
- **Add re-ranking** (Cohere Rerank) to improve retrieval precision
- **Expose as FastAPI service** — see `app/main.py` in the repo
- **Add evaluation** using RAGAS framework (faithfulness, answer relevancy)
- **Multi-document RAG** — index multiple RBI circulars or annual reports